In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os

# System definitions

In [ ]:
from nsflows.systems.gaussians import normal
from nsflows.systems.testsystems_2D import double_well

n_particles = 1
dimensions = 2

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

normal_2D = normal(n_particles=n_particles, dimensions=dimensions, device=device)
double_well_2D = double_well(n_particles=n_particles, dimensions=dimensions, device=device, eps=3., c=1., d=0.5)

# Plot PES

In [ ]:
x_min = -2.1
x_max = +2.1
y_min = -4.0
y_max = +3.5
n_grid = 200

x = np.linspace(x_min, x_max, n_grid)
y = np.linspace(y_min, y_max, n_grid)

X, Y = np.meshgrid(x, y)

Z_source = np.zeros([len(X),len(Y)])
Z_target = np.zeros([len(X),len(Y)])

for i in range(len(X)):
    for j in range(len(Y)):

        conf = torch.from_numpy(np.array([X[i][j], Y[i][j]], dtype=np.float32)).unsqueeze(0).to(device)

        Z_source[i,j] = normal_2D.energy(conf).squeeze().cpu().numpy()
        Z_target[i,j] = double_well_2D.energy(conf).squeeze().cpu().numpy()

In [ ]:
fig_size = (10 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400)


# ax.scatter(cpu_samples[:,0], cpu_samples[:,1], s=.25, zorder = 10)
ax.contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
# ax.contour(X, Y, Z_target, levels=[U_max], colors="C1", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xticks([])  # Remove x ticks
ax.set_yticks([])  # Remove y ticks

plt.show()

## Define Parameters

In [ ]:
# Load Previous Training
load = False

# Nested Sampling Parameters
live_samples = 10000

# The initial live-sample set is picked from data/dw/ according to live_samples.
init_samples_dir = "../data/dw"
init_samples_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", "samples_init.pt")
if not os.path.exists(init_samples_filepath):
    available = sorted(d[1:].lstrip("0") for d in os.listdir(init_samples_dir) if d.startswith("K"))
    raise FileNotFoundError(
        f"No initial samples shipped for live_samples={live_samples} "
        f"(looked for {init_samples_filepath}); available: {available}"
    )

max_ns_iterations = 100000
n_propagate = 1000
update_step = True
turn_on_nf = 100
alternate_std_ns_iters = 100
n_pool = 20000
load_nf_parameters = False
itrain = 4
reinitialize_nf_parameters = False
cumulate_n_dataset = 3

# Network Parameters
conditioned = True
n_blocks = 2
n_bins = 8

max_lr = 2.5e-4
mid_lr = 1e-4 
start_lr = max_lr/25
end_lr = 1e-6

# Training Parameters
training_protocol = [

    {
        "w_xz" : 1,
        "w_zx" : 0,
        "batch_size" : 1000,
        "conds_per_batch" : -1,
        "total_steps" : 750,
        "start_lr" : start_lr,
        "end_lr" : mid_lr,
        "max_lr" : max_lr,
        "save_best" : True,
        "optimizer" : "Adam",
        "scheduler" : "OneCycleLR",
    }, 

    {
        "w_xz" : 1,
        "w_zx" : 0,
        "batch_size" : 1000,
        "conds_per_batch" : -1,
        "total_steps" : 250,
        "start_lr" : mid_lr,
        "end_lr" : end_lr,
        "max_lr" : None,
        "save_best" : True,
        "optimizer" : "Adam",
        "scheduler" : "CosineAnnealingLR",
    },  
]


# Output Folder Definition

In [ ]:
from nsflows.tools.util import generate_unique_identifier, remove_empty_directories, generate_output_directory

remove_empty_directories("./output/")

if load:
    output_dir = "./output/..." # Insert output folder here if you want to load from an existing run. Otherwise, the code will generate a new run folder.
    print(f"Run Folder: {output_dir}")
else:
    run_id = generate_unique_identifier()
    output_dir = generate_output_directory(run_id)

# Flow definition

In [ ]:
import copy

from nsflows.network.flow_assembler import flow_assembler
from nsflows.network.coupling_blocks import EquivariantRQS
from nsflows.transformations.normalization import NormalizeBox

block_unit = [
    
    EquivariantRQS((0,), n_particles, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    EquivariantRQS((1,), n_particles, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    
    EquivariantRQS((1,), n_particles, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    EquivariantRQS((0,), n_particles, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
]
block_list = [copy.deepcopy(element) for block in range(n_blocks) for element in block_unit]

box_pr = torch.from_numpy(np.array([2*10, 2*10], dtype=np.float32)).to(device)
box_sys = torch.from_numpy(np.array([2*10, 2*10], dtype=np.float32)).to(device)

norm_box_pr = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_pr, device=device)
norm_box_sys = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_sys, device=device)

# Initialization of the flow  
flow = flow_assembler(normal_2D, double_well_2D, device=device, 
                      blocks = block_list,
                      prior_sided_transformation_layers = [norm_box_pr], 
                      post_sided_transformation_layers = [norm_box_sys],
                      j=0,
                    ).to(device)

flow_parameters = sum(p.numel() for p in flow.parameters() if p.requires_grad)
print(f"Network parameters: {flow_parameters}")

In [ ]:
print(flow)

# Nested Sampling

In [ ]:
from nsflows.samplers.monte_carlo import rejection_monte_carlo
from nsflows.samplers.DL_samplers import nflows_propagator
from nsflows.nested_sampling import nested_sampling

rejection_sampler = rejection_monte_carlo(system=double_well_2D, n_cycles=100, step_size=1.2, transform=False)
nflows_sampler = nflows_propagator(flow, conditioned, transform=False)

if load:
    acceptance, umax_plt = np.loadtxt(os.path.join(output_dir, "output.txt"), usecols=(2,3), unpack=True)
    samples = torch.load(os.path.join(output_dir, "samples.pt"))
    U_samples = umax_plt[-1]

    count = 0
    while True:
        if training_protocol[-1]["save_best"]:
            flow_parameters_filepath = os.path.join(output_dir, f"best_flow_parameters_{count}.pt")
        else:
            flow_parameters_filepath = os.path.join(output_dir, f"flow_parameters_{count}.pt")
        if not os.path.exists(flow_parameters_filepath):
            print(f"File {flow_parameters_filepath} not found")
            break
        print(f"Loading network parameters from {flow_parameters_filepath}")
        nflows_sampler.flow.load_state_dict(torch.load(flow_parameters_filepath))
        count += 1
else:
    samples, U_samples, acceptance, umax_plt = nested_sampling(K=live_samples, 
                                                            system=double_well_2D, 
                                                            std_propagator=rejection_sampler, 
                                                            nf_propagator=nflows_sampler, 
                                                            init_samples_filepath=init_samples_filepath, 
                                                            max_iters=max_ns_iterations, 
                                                            n_propagate=n_propagate, 
                                                            update_step=update_step,
                                                            turn_on_nf=turn_on_nf, 
                                                            alternate_std_ns_iters=alternate_std_ns_iters, 
                                                            n_pool=n_pool, 
                                                            load_nf_parameters=load_nf_parameters, 
                                                            reinitialize_nf_parameters=reinitialize_nf_parameters,
                                                            itrain=itrain,
                                                            cumulate_n_dataset=cumulate_n_dataset,
                                                            training_protocol=training_protocol,
                                                            iprint=1, 
                                                            isavesamp=500,
                                                            save_biased_pool=True,
                                                            outputdir=output_dir)

# Plot Output

## Training Metrics

In [ ]:
import re
import glob

# Regex patterns for possible filename formats
patterns = [
    re.compile(r"train_log_(\d+)\.txt$"),               # train_log_count.txt OR train_log_stage.txt
    re.compile(r"train_log_(\d+)_(\d+)\.txt$"),         # train_log_count_stage.txt
]

all_metrics = []

# Find all candidate files
for filepath in sorted(glob.glob(os.path.join(output_dir, "train_log_*.txt"))):
    filename = os.path.basename(filepath)
    stage = count = None

    # Try matching each pattern
    for pat in patterns:
        match = pat.match(filename)
        if match:
            groups = match.groups()
            if len(groups) == 1:
                # Could be count-only or stage-only — label generically
                count = int(groups[0])
            elif len(groups) == 2:
                count, stage = map(int, groups)
            break

    if not match:
        # Skip anything that doesn’t match expected formats
        continue

    try:
        print(f"Loaded: {filename} (count={count}, stage={stage})")
        metrics = np.loadtxt(filepath)
    except Exception as e:
        print(f"Could not load {filename}: {e}")

    length_x = 60
    fig_size = (length_x * 0.393701, 10 * 0.393701)
    fig, ax = plt.subplots(1, 5, figsize = fig_size, dpi = 400, tight_layout=True)

    ax[0].plot(metrics[:,0], metrics[:,2], label="train")
    ax[0].plot(metrics[:,0], metrics[:,6], label="eval")
    ax[0].set_xlabel("epochs")
    ax[0].set_ylabel("NLL loss")

    ax[1].plot(metrics[:,0], metrics[:,3], label="train")
    ax[1].plot(metrics[:,0], metrics[:,7], label="eval")
    ax[1].set_xlabel("epochs")
    ax[1].set_ylabel("ECUT loss")

    ax[2].plot(metrics[:,0], metrics[:,4], label="train")
    ax[2].plot(metrics[:,0], metrics[:,8], label="eval")
    ax[2].set_xlabel("epochs")
    ax[2].set_ylabel("ECUT violation")
    ax[2].legend(frameon=False)
    # ax[2].set_yscale("log")

    ax[3].plot(metrics[:,0], metrics[:,5], color="C0", label="train")
    ax[3].set_xlabel("epochs")
    ax[3].set_ylabel("Gradient Norm")
    # ax[3].set_ylim(0, 1)
    ax[3].legend(frameon=False)
    ax[3].set_yscale("log")

    ax[4].plot(metrics[:,0], metrics[:,9], color="C1", label="eval")
    ax[4].set_xlabel("epochs")
    ax[4].set_ylabel("RESS")
    ax[4].set_ylim(0, 1)
    ax[4].legend(frameon=False)
    
    # plt.savefig(os.path.join(output_dir, "metrics.png"))
    plt.show()

## Reference, Generated, Resampled

In [ ]:
count = 0
while True:
    
    dataset_filepath = os.path.join(output_dir, f"dataset_{count:04d}.pt")
    pool_biased_filepath = os.path.join(output_dir, f"pool_biased_{count:04d}.pt")
    pool_filepath = os.path.join(output_dir, f"pool_{count:04d}.pt")
    conds_filepath = os.path.join(output_dir, f"conds_{count:04d}.pt")
    if (not os.path.exists(dataset_filepath)) or (not os.path.exists(pool_biased_filepath)) or (not os.path.exists(pool_filepath)) or (not os.path.exists(conds_filepath)) :
        print(f"File {dataset_filepath} " + (f"found" if os.path.exists(dataset_filepath) else f"not found"))
        print(f"File {pool_biased_filepath} " + (f"found" if os.path.exists(pool_biased_filepath) else f"not found"))
        print(f"File {pool_filepath} " + (f"found" if os.path.exists(pool_filepath) else f"not found"))
        print(f"File {conds_filepath} " + (f"found" if os.path.exists(conds_filepath) else f"not found"))
        break
    count += 1

    dataset = torch.load(dataset_filepath)
    pool_biased = torch.load(pool_biased_filepath)
    pool = torch.load(pool_filepath)
    cond = torch.load(conds_filepath)

    dataset_cpu = dataset.view(-1, double_well_2D.n_particles, double_well_2D.dimensions).cpu().numpy()
    pool_biased_cpu = pool_biased.view(-1, double_well_2D.n_particles, double_well_2D.dimensions).cpu().numpy()
    pool_cpu = pool.view(-1, double_well_2D.n_particles, double_well_2D.dimensions).cpu().numpy()
    
    fig_size = (30 * 0.393701, 7 * 0.393701)
    fig, ax = plt.subplots(1, 3, figsize = fig_size, dpi = 400)

    ax[0].scatter(dataset_cpu[:,:,0], dataset_cpu[:,:,1], s=.25, zorder = 10, alpha=1, color="C0")
    ax[0].contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
    ax[0].contour(X, Y, Z_target, levels=[cond.item()], colors="C4", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
    ax[0].set_xlim(x_min, x_max)
    ax[0].set_ylim(y_min, y_max)

    ax[1].scatter(pool_biased_cpu[:,:,0], pool_biased_cpu[:,:,1], s=.25, zorder = 10, alpha=1, color="C2")
    ax[1].contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
    ax[1].contour(X, Y, Z_target, levels=[cond.item()], colors="C4", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
    ax[1].set_xlim(x_min, x_max)
    ax[1].set_ylim(y_min, y_max)

    ax[2].scatter(pool_cpu[:,:,0], pool_cpu[:,:,1], s=.25, zorder = 10, alpha=1, color="C3")
    ax[2].contour(X, Y, Z_target-Z_target.min(), levels=np.arange(0,20,.5), cmap = "Greys_r", alpha = 1, linewidths=1, zorder = 0)
    ax[2].contour(X, Y, Z_target, levels=[cond.item()], colors="C4", alpha = 1, linewidths=1, linestyles='-', zorder = 10)
    ax[2].set_xlim(x_min, x_max)
    ax[2].set_ylim(y_min, y_max)


## Generation Efficiency

In [ ]:
gen_attempts = []
count = 0 
while True:
    generation_log_filepath = os.path.join(output_dir, f"generation_log_{count:04d}.txt")
    if not os.path.exists(generation_log_filepath):
        print(f"File {generation_log_filepath} not found")
        break
    count += 1
    gen_attempts.append(np.loadtxt(generation_log_filepath, usecols=(0), unpack=True)[-1])

fig_size = (10 * 0.393701, 7.5 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 600)

ax.set_yscale("log")
ax.plot(np.array(gen_attempts) + 1, ls="-")

ax.set_xlabel(r"Pool generated")
ax.set_ylabel(r"Generation attempts")

plt.show()

## Energy vs. Iteration and Density of States vs. Iteration

In [1]:
# Reference standard-NS run, for the same number of live samples as the run above.
reference_filepath = os.path.join(init_samples_dir, f"K{live_samples:05d}", "reference_from_std_ns.txt")
umax_plt_std = np.loadtxt(reference_filepath, unpack=True, usecols=3)

vals_std, bins_std = np.histogram(umax_plt_std[100:]-umax_plt_std[100:].min(), bins=200)
vals, bins = np.histogram(umax_plt-umax_plt.min(), bins=200)
bin_centers = .5*(bins[1:] + bins[:-1])
bin_centers_std = .5*(bins_std[1:] + bins_std[:-1])

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.lines as mlines

fig_size = (10 * 0.393701, 8 * 0.393701)
fig, ax = plt.subplots(1, 2, figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

ax[0].plot(np.linspace(0, 10, len(umax_plt)), (umax_plt-umax_plt.min()), color="C3")
ax[0].plot(np.linspace(0, 10, len(umax_plt_std[100:])), (umax_plt_std[100:]-umax_plt_std[100:].min()), color="C0", ls=":")
ax[0].set_ylabel("Energy")
ax[0].set_xlabel(r"Iteration ($\times 10^4$)")

ax_inset = inset_axes(ax[0], width="50%", height="30%", loc='upper right', borderpad=.5)
ax_inset.plot(np.linspace(0, 10, len(umax_plt)), (umax_plt-umax_plt.min()), color="C3")
ax_inset.plot(np.linspace(0, 10, len(umax_plt_std[100:])), (umax_plt_std[100:]-umax_plt_std[100:].min()), color="C0", ls=":")

ax_inset.set_yscale("log")
ax_inset.set_ylim(.9e-2,1.6e2)
lines_ls = []
lines_ls.append(mlines.Line2D([], [], color='C3', linestyle='-', label=r'Energy NFs NS'))
lines_ls.append(mlines.Line2D([], [], color='C0', linestyle=':', label=r'Energy std. NS'))


# Second legend (line styles) outside plot (right side)
ax[0].legend(handles=lines_ls, loc='lower center',
                    bbox_to_anchor=(0.5, 1.02), ncol=1, frameon=False)

ax[1].plot(vals, bin_centers, color="C1")
ax[1].plot(vals_std, bin_centers_std, color="C4", ls=":")
ax[1].set_xlabel(r"Samples")
ax[1].set_xscale('log')

# Proxies for styled black lines
lines_ls = []
lines_ls.append(mlines.Line2D([], [], color='C1', linestyle='-', label=r'DoS NFs NS'))
lines_ls.append(mlines.Line2D([], [], color='C4', linestyle=':', label=r'DoS std. NS'))

# Second legend (line styles) outside plot (right side)
ax[1].legend(handles=lines_ls, loc='lower center',
                    bbox_to_anchor=(0.5, 1.02), ncol=1, frameon=False)

plt.show()

NameError: name 'os' is not defined

## Timings

In [ ]:
data = np.genfromtxt(os.path.join(output_dir, "timings.txt"), skip_header=2, names=True, comments="#")
timings = np.column_stack([data[name] for name in data.dtype.names])

# Labels for tasks
labels = data.dtype.names

fig_size = (15 * 0.393701, 7 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 400, sharey=True, tight_layout=True)

x = np.arange(timings.shape[0])  # one bar per row
bottom = np.zeros(timings.shape[0])

labels = [r"Standard MCMC ($10^2$ NS steps)", r"Selection from Pool (~ $10^4$ NS steps)", "Network Training", "Pool Generation"]
for i, label in enumerate(labels):
    ax.bar(x+1, timings[:, i], bottom=bottom, label=label)
    bottom += np.nan_to_num(timings[:, i])  # accumulate heights, treating NaN as 0

ax.set_xlabel("Pool generated")
ax.set_ylabel("Time (seconds)")
# ax.set_title("Task timings per sample")
# ax.legend(title="Tasks")

# plt.legend()
ax.legend(loc='lower center',
                    bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

plt.tight_layout()
plt.show()